# Excel Data Ingestion with LakeLogic 📊

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/files/excel/excel_example.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/files/excel/excel_example.ipynb)

## Business Scenario
Your finance team sends you monthly employee data reports in Excel format. These spreadsheets contain:
- Employee contact information
- Salary data
- Department assignments
- Employment status

**The Problem**: Manual Excel uploads often contain data quality issues that break downstream reports.

**The Solution**: Use LakeLogic to automatically validate Excel files before they reach your data warehouse.

## Value Proposition
- **Zero Manual Parsing**: Direct `.xlsx` and `.xls` file reading
- **Instant Validation**: Catch errors before they impact analytics
- **User-Friendly Error Reports**: Send clear feedback to data providers
- **Automated Pipeline**: Transform manual uploads into governed data flows

## 1. Setup
Install LakeLogic with Excel support and set up your environment.

In [ ]:
# Uncomment to install LakeLogic with Excel support
# %pip install lakelogic[pandas]  # includes openpyxl for Excel

import os
from pathlib import Path
from lakelogic import DataProcessor

# Set engine preference
os.environ["LAKELOGIC_ENGINE"] = "pandas"  # Best for Excel files

print("✅ LakeLogic ready for Excel ingestion!")

## 2. Preview Excel File Metadata
Let's check what sheets and data the Excel file contains.

In [ ]:
import pandas as pd

excel_path = "data/employees.xlsx"

# Load Excel file to inspect
excel_file = pd.ExcelFile(excel_path)
print(f"📁 Excel File: {excel_path}")
print(f"📄 Sheet Names: {excel_file.sheet_names}")
print(f"\n📊 Preview of first sheet:")
preview_df = pd.read_excel(excel_path, nrows=3)
print(preview_df)

## 3. Run LakeLogic Validation
Now let's use LakeLogic to validate the Excel data against our contract.

In [ ]:
# Initialize processor
processor = DataProcessor(contract="excel_contract.yaml")

# Run ingestion - auto-detects .xlsx extension
result = processor.run_source("data/employees.xlsx")

print("\n" + "="*50)
print("         EXCEL VALIDATION SUMMARY")
print("="*50)
print(f"📥 Total records loaded:  {len(result.raw)}")
print(f"✅ Valid records:         {len(result.good)}")
print(f"❌ Invalid records:       {len(result.bad)}")
print("="*50)

## 4. Access Results Using Named Attributes
LakeLogic returns a `ValidationResult` object with clear, named attributes.

In [ ]:
# Using the recommended named attribute approach
original_excel_data = result.raw
validated_employees = result.good
quarantined_employees = result.bad

print("📊 Result object:")
print(result)
print(f"\n✅ Validated data ready for warehouse: {len(validated_employees)} records")
print(f"🚨 Data quality issues to review: {len(quarantined_employees)} records")

## 5. Inspect Validated Data
Let's see the clean data that passed all validation rules.

In [ ]:
print("✅ VALIDATED EMPLOYEES (Ready for Analytics):")
print(validated_employees)

if len(validated_employees) > 0:
    print(f"\n📋 Columns: {list(validated_employees.columns)}")
    print(f"\n💰 Average Salary: ${validated_employees['salary'].mean():,.2f}")
    print(f"\n🏢 Departments represented: {validated_employees['department'].nunique()}")

## 6. Review Data Quality Issues
For quarantined records, let's see exactly what went wrong.

In [ ]:
print("🚨 DATA QUALITY ISSUES:")

if len(quarantined_employees) > 0:
    print(quarantined_employees[['id', 'name', 'email', 'salary', 'status', '_lakelogic_errors']])
    
    print("\n📋 Detailed Error Report:")
    print("-" * 60)
    
    for idx, row in quarantined_employees.iterrows():
        print(f"\n❌ Employee: {row['name']} (ID: {row['id']})")
        errors = row['_lakelogic_errors']
        for error in errors:
            print(f"   • {error}")
else:
    print("🎉 No data quality issues! All records are valid.")

## 7. Generate User-Friendly Error Report
Create a simple report to send back to your data providers.

In [ ]:
if len(quarantined_employees) > 0:
    error_report = quarantined_employees[['id', 'name', 'email', 'salary', 'status']].copy()
    error_report['issues'] = quarantined_employees['_lakelogic_errors'].apply(lambda x: '; '.join(x))
    
    print("📧 Error Report for Data Provider:")
    print("=" * 80)
    print("The following records from your Excel file have data quality issues:")
    print()
    print(error_report.to_string(index=False))
    print("=" * 80)
    print("\nPlease correct these issues and resubmit the file.")
    
    # Optionally save to CSV for email attachment
    error_report.to_csv("data_quality_issues.csv", index=False)
    print("\n💾 Error report saved to: data_quality_issues.csv")
else:
    print("✅ No errors to report - file is ready for processing!")

## 8. Calculate Data Quality Metrics
Track your data quality over time with key metrics.

In [ ]:
total_records = len(result.raw)
valid_records = len(result.good)
invalid_records = len(result.bad)

pass_rate = (valid_records / total_records * 100) if total_records > 0 else 0
fail_rate = (invalid_records / total_records * 100) if total_records > 0 else 0

print("\n" + "="*50)
print("          DATA QUALITY SCORECARD")
print("="*50)
print(f"Total Records Processed:    {total_records:>6}")
print(f"Successfully Validated:     {valid_records:>6} ({pass_rate:>5.1f}%)")
print(f"Failed Validation:          {invalid_records:>6} ({fail_rate:>5.1f}%)")
print("="*50)

# Quality grade
if pass_rate >= 95:
    grade = "🌟 EXCELLENT"
elif pass_rate >= 80:
    grade = "✅ GOOD"
elif pass_rate >= 60:
    grade = "⚠️  NEEDS IMPROVEMENT"
else:
    grade = "❌ CRITICAL - REVIEW REQUIRED"

print(f"\nData Quality Grade: {grade}")

## 9. Multi-Sheet Excel Files (Advanced)
Working with Excel files that have multiple sheets.

In [ ]:
# Example: Processing a specific sheet from a multi-sheet workbook
# If your Excel file has multiple sheets, you can specify which one to read

# Option 1: Pre-load specific sheet with pandas
# specific_sheet = pd.read_excel("data/multi_sheet.xlsx", sheet_name="Employees")
# result = processor.run(specific_sheet)

# Option 2: Process all sheets in a loop
# excel_file = pd.ExcelFile("data/multi_sheet.xlsx")
# for sheet_name in excel_file.sheet_names:
#     df = pd.read_excel(excel_file, sheet_name=sheet_name)
#     result = processor.run(df)
#     print(f"Sheet '{sheet_name}': {result}")

print("💡 TIP: For multi-sheet workbooks, pre-load the specific sheet")
print("    with pandas.read_excel() and pass it to processor.run()")

## Summary

In this tutorial, we demonstrated:

1. ✅ **Native Excel Reading**: Direct `.xlsx` file ingestion without manual conversion
2. ✅ **Named Attributes**: Clean API with `result.good`, `result.bad`, `result.raw`
3. ✅ **Validation Rules**: Email formats, salary ranges, and status checks
4. ✅ **Error Reporting**: Generate user-friendly reports for data providers
5. ✅ **Quality Metrics**: Track data quality with actionable scorecards

### Production Use Cases

**Finance Team Uploads**
```python
result = processor.run_source("monthly_budget.xlsx")
if len(result.bad) > 0:
    send_email_to_finance(result.bad)
else:
    warehouse.write_table("finance.budget", result.good)
```

**HR Data Integration**
```python
result = processor.run_source("employee_updates.xlsx")
metrics = {"pass_rate": len(result.good) / len(result.raw) * 100}
dashboard.update_quality_metrics(metrics)
```

### Next Steps

- **Customize Validation**: Edit `excel_contract.yaml` to match your business rules
- **Automate**: Build a scheduled job to process Excel uploads from a shared folder
- **Materialize**: Use LakeLogic's `materialize` option to save validated data
- **Monitor**: Track data quality trends over time